<a href="https://github.com/N3kooz/Segsmaker_Mods">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

*   get your civitai api key from [here](https://civitai.com/user/account)

In [ ]:
# @title <b><font color='orange'>WebUI Installer</font></b> {"display-mode":"form"}

Webui = 'Forge' # @param ["A1111", "Forge", "ReForge", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai___Key = '' # @param { type: "string", placeholder: "Your Civitai API Key (required)" }
HF_Read_Token = '' # @param { type: "string", placeholder: "Your Huggingface READ Token (optional)" }
Mount__GDrive = 'No' # @param ["Yes", "No"]

mount = Mount__GDrive

if mount == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

!curl -sLo /content/setup.py https://github.com/N3kooz/Segsmaker_Mods/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai___Key" --hf_read_token="$HF_Read_Token"

if mount == 'Yes':
    from pathlib import Path

    d = Path('/content/drive/MyDrive/Segsmaker')

    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        f = d / n
        f.mkdir(parents=True, exist_ok=True)
        s = p / f'drive-{n}'
        s.symlink_to(f, target_is_directory=True)

    !rm -rf $WebUI_Output
    o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    o.mkdir(parents=True, exist_ok=True)
    WebUI_Output.symlink_to(o, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        wc = WebUI / 'cache'
        !rm -rf $wc
        c = d / 'cache'
        c.mkdir(parents=True, exist_ok=True)
        wc.symlink_to(c, target_is_directory=True)

In [ ]:
# @title  Pengaturan Download Model & Ekstensi
# @markdown Isi link di bawah ini sesuai kebutuhan. Kosongkan jika tidak ingin mendownload.

# @markdown ---
# @markdown ###  Extensions (GitHub Link)
extension_link = "" # @param {type:"string"}

# @markdown ###  VAE & Embeddings
vae_link = "" # @param {type:"string"}
embedding_link = "" # @param {type:"string"}

# @markdown ###  FLUX Components
unet_link = "" # @param {type:"string"}
clip_link = "" # @param {type:"string"}

# --- LOGIKA KODE (Akan berjalan otomatis) ---

if extension_link:
    %cd -q $Extensions
    !git clone {extension_link}

if vae_link:
    %cd -q $VAE
    %download {vae_link}

if embedding_link:
    %cd -q $Embeddings
    %download {embedding_link}

if unet_link:
    %cd -q $UNET
    %download {unet_link}

if clip_link:
    %cd -q $CLIP
    %download {clip_link}

In [ ]:
# @title  Model & LoRA Downloader (5 Slots)
# @markdown Isi link di bawah ini. Kosongkan slot yang tidak digunakan.

# --- SLOTS CONFIGURATION ---

# @markdown ###  Checkpoint Slots
ckpt_1 = "" # @param {type:"string"}
ckpt_2 = "" # @param {type:"string"}
ckpt_3 = "" # @param {type:"string"}
ckpt_4 = "" # @param {type:"string"}
ckpt_5 = "" # @param {type:"string"}

# @markdown ###  LoRA Slots
lora_1 = "" # @param {type:"string"}
lora_2 = "" # @param {type:"string"}
lora_3 = "" # @param {type:"string"}
lora_4 = "" # @param {type:"string"}
lora_5 = "" # @param {type:"string"}

# --- EXECUTION ---

# Daftar link untuk diproses
checkpoints = [ckpt_1, ckpt_2, ckpt_3, ckpt_4, ckpt_5]
loras = [lora_1, lora_2, lora_3, lora_4, lora_5]

# Proses Checkpoints
%cd -q $CKPT
print(" Sedang mengunduh Checkpoints...")
for link in checkpoints:
    if link.strip():
        %download {link}

# Proses LoRAs
%cd -q $LORA
print("\n Sedang mengunduh LoRAs...")
for link in loras:
    if link.strip():
        %download {link}

print("\n Semua proses selesai!")

In [ ]:


''' Controlnet '''
%run $Controlnet_Widget

In [ ]:
# @title 🚀 Launcher WebUI { display-mode: "form" }
# @markdown Pilih software dan masukkan token tunnel jika diperlukan.

# --- UI COMPONENTS ---
Software = "Forge" # @param ["A1111", "Forge", "ReForge", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Ngrok_Token = "2SrmI5xCy7MGdmpvSym3fTk5qEs_jDAWhnE2Y9pQof1mejaZ" # @param {type:"string"}
Zrok_Token = "" # @param {type:"string"}

def launch():
    # Mapping Arguments
    args_map = {
        "A1111": "--xformers",
        "Forge": "--disable-xformers --opt-sdp-attention --cuda-stream",
        "ReForge": "--xformers --cuda-stream",
        "Forge-Classic": "--xformers --cuda-stream --persistent-patches",
        "Forge-Neo": "--xformers --cuda-stream",
        "ComfyUI": "--dont-print-server --use-pytorch-cross-attention --skip-comfyui-check",
        "SwarmUI": "--launch_mode none"
    }

    selected_args = args_map.get(Software, "")

    # Add Tunnels
    if Ngrok_Token.strip():
        selected_args += f" --N={Ngrok_Token.strip()}"

    if Zrok_Token.strip():
        selected_args += f" --Z={Zrok_Token.strip()}"

    # Execution
    import os
    print(f"📦 Launching {Software} with args: {selected_args}")

    %cd -q $WebUI
    %run segsmaker.py {selected_args}

launch()

## Launcher
args list :
-  **A1111** = `--xformers`
- **Forge** = `--disable-xformers --opt-sdp-attention --cuda-stream`
- **ReForge** = `--xformers --cuda-stream`
- **Forge-Classic** = `--xformers --cuda-stream --persistent-patches`
- **Forge-Neo** = `--xformers --cuda-stream`
- **ComfyUI** = `--dont-print-server --use-pytorch-cross-attention`
- **SwarmUI** = `--launch_mode none`
<br><br>

For ComfyUI, add `--skip-comfyui-check` to skip checking the main requirements and custom node dependencies

Add **--N=ngrok_token** to start NGROK tunnel<br>
Add **--Z=zrok_token** to start ZROK tunnel